 # Experiment 3 — Null Models (Revised)

 Conditions (6 total, all analysed together):
  1. Video model                             (real trained pipeline)
  2. Shuffled-pairing model, original-target scoring
  3. Shuffled-pairing model, permuted-target scoring
  4. Training-set mean waveform
  5. Training-mean constant
  6. Zero-output baseline

The shuffled-pairing model is a single trained pipeline. Its held-out
predictions are evaluated against two different EEG reference arrays:
the original trial-matched targets (original-target scoring) and the
targets assigned by the test-split permutation (permuted-target
scoring). Because these evaluations use different reference arrays,
their condition-level estimates are reported separately and are not
interpreted as a like-for-like comparison.

 Statistical hierarchy (identical to Section 4.4):
   - 40 held-out trials    : repeated-measures unit within participant
   - 18 participants       : independent unit at population level

 Aggregation contract (identical to Section 4.4):
   - Trial-level Pearson r → Fisher-z transform → equal-weight mean
     across trials → back-transform per participant (r_pooled).
   - Population-level r    → Fisher-z transform across the 18
     participant-level r_pooled values → equal-weight mean →
     back-transform.
   - 95% CI                → percentile bootstrap over the 18
     independent participants, resampled on the Fisher-z scale,
     back-transformed for display.

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from statsmodels.stats.multitest import multipletests

sys.path.insert(0, r"D:\Thesis\Codes\Source")

from face2eeg.core.results import load_experiment_result
from face2eeg.core.stats import (
    summarize_trials,
    compare_methods_trial_level,
    fisher_z,
    inv_fisher_z,
    sign_flip_pvalue,
    population_cohens_dz,
)
from face2eeg.analysis import validate_trial_alignment
from face2eeg.plotting import plot_null_models_summary
from face2eeg.core.config import VALID_SUBJECTS, SEED

%matplotlib inline

## 1. Configuration

In [ ]:
BASE = Path(r"D:\Thesis\Codes\Data\Experiments\Cross-Validation")

MAIN_RES_DIR     = BASE / "Between-Participant-Variability" / "Results"
NULL_DIR         = BASE / "Null-Models"
SHUFFLED_RES_DIR = NULL_DIR / "Shuffled" / "Results"
TRIVIAL_RES_DIR  = NULL_DIR / "Trivial" / "Results"

FIG_DIR   = NULL_DIR / "Analysis-Figures"
TABLE_DIR = NULL_DIR / "Analysis-Tables"
FIG_DIR.mkdir(parents=True, exist_ok=True)
TABLE_DIR.mkdir(parents=True, exist_ok=True)

SUBJECTS = list(VALID_SUBJECTS)
EXPECTED_SUBJECTS = [1, 2, 4, 6, 7, 8, 9, 10, 12, 13,
                     15, 16, 17, 18, 19, 20, 21, 22]

N_TRIALS               = 40
N_FOLDS                = 5
N_TEST_TRIALS_PER_FOLD = 8

EEG_FS                 = 128
TRIAL_DURATION         = 60
EXPECTED_TRIAL_SAMPLES = EEG_FS * TRIAL_DURATION   # 7680

CHANNEL     = "Fz"
TIME_WINDOW = 2

N_BOOT   = 10_000
N_PERM   = 10_000
CI_LEVEL = 0.95

# ---- Six named conditions (Video model == Proposed model) ----
CONDITIONS = [
    "Video model",
    "Shuffled-pairing model, original-target scoring",
    "Shuffled-pairing model, permuted-target scoring",
    "Training-set mean waveform",
    "Training-mean constant",
    "Zero-output baseline",
]

# Map each condition to (prediction_key, target_key)
#   target_key = "original" → score against the ORIGINAL trial-matched EEG targets
#   target_key = "permuted" → score against the EEG targets assigned by the
#                             test-split permutation (permuted-target scoring)
CONDITION_SPEC = {
    "Video model":
        ("Video model",                "original"),
    "Shuffled-pairing model, original-target scoring":
        ("Shuffled model",             "original"),
    "Shuffled-pairing model, permuted-target scoring":
        ("Shuffled model",             "permuted"),
    "Training-set mean waveform":
        ("Training-set mean waveform", "original"),
    "Training-mean constant":
        ("Training-mean constant",     "original"),
    "Zero-output baseline":
        ("Zero-output baseline",       "original"),
}

# Reference level for one-sample RMSE tests
RMSE_REF = 1.0

TRIVIAL_KEYS = {
    "Training-set mean waveform": "mean_waveform",
    "Training-mean constant":     "const_train_mean",
    "Zero-output baseline":       "const_zero",
}

assert SUBJECTS == EXPECTED_SUBJECTS
assert len(SUBJECTS) == 18

print(f"Subjects      : {len(SUBJECTS)}")
print(f"Channel       : {CHANNEL}")
print(f"T (s)         : {TIME_WINDOW}")
print(f"Bootstrap     : {N_BOOT} resamples (percentile)")
print(f"Sign-flip     : {N_PERM} permutations")
print(f"Seed          : {SEED}")
print(f"Conditions    : {CONDITIONS}")

## 2. Input-file audit

In [ ]:
main_files = {s: MAIN_RES_DIR     / f"sub{s:02d}_chFz.pkl" for s in SUBJECTS}
shuf_files = {s: SHUFFLED_RES_DIR / f"sub{s:02d}_chFz.pkl" for s in SUBJECTS}
triv_files = {s: TRIVIAL_RES_DIR  / f"sub{s:02d}_chFz.pkl" for s in SUBJECTS}

for name, files in {
    "Video model":       main_files,
    "Shuffled-pairing model":          shuf_files,
    "Trivial baselines": triv_files,
}.items():
    missing = [str(p) for p in files.values() if not p.exists()]
    if missing:
        raise FileNotFoundError(
            f"{name}: {len(missing)} missing file(s):\n" + "\n".join(missing)
        )

print(f"All required files exist for {len(SUBJECTS)} subjects.")


## 3. Metadata and cross-validation audit

In [ ]:
audit_result = load_experiment_result(str(main_files[SUBJECTS[0]]))

print("ExperimentResult metadata")
print("-------------------------")
print("subject    :", audit_result.subject)
print("channel    :", audit_result.channel)
print("time_window:", audit_result.time_window)
print("n_splits   :", audit_result.n_splits)
print("pred_key   :", audit_result.pred_key)
print("folds      :", audit_result.fold_ids())

In [ ]:
for subject in SUBJECTS:
    result = load_experiment_result(str(main_files[subject]))

    assert result.channel == CHANNEL
    assert float(result.time_window) == float(TIME_WINDOW)
    assert result.n_splits == N_FOLDS

    folds = result.fold_ids()
    assert len(folds) == N_FOLDS

    for fold_idx in folds:
        y_fold, _ = result.get_fold_trials(fold_idx)
        assert y_fold.shape[0] == N_TEST_TRIALS_PER_FOLD

print("CV structure verified: 5 folds × 8 held-out trials.")

## 4. Load prediction results
Predictions are stored under pipeline keys:
   - "Video model"                (real trained)
   - "Shuffled model"             (shuffled-trained, shared for both evaluations)
   - "Training-set mean waveform"
   - "Training-mean constant"
   - "Zero-output baseline"

 The condition-name → (pred_key, target_key) mapping is defined in
 CONDITION_SPEC.

In [ ]:
results = {}

for subject in SUBJECTS:

    # ---- Video model ----
    r_main = load_experiment_result(str(main_files[subject]))
    y_true, y_pred_main = r_main.get_all_trials()
    trial_ids_main = r_main.get_trial_indices()

    # ---- Shuffled model (predictions shared;
#      targets assigned by the test-split permutation stored) ----
    r_shuf = load_experiment_result(str(shuf_files[subject]))
    y_true_shuf_permuted, y_pred_shuf = r_shuf.get_all_trials()
    trial_ids_shuf = r_shuf.get_trial_indices()

    validate_trial_alignment(
        trial_ids_main, trial_ids_shuf,
        name_a="Video model", name_b="Shuffled-pairing model",
    )

    # ---- Trivial baselines ----
    trivial_preds = {}
    for label, pred_key in TRIVIAL_KEYS.items():
        r_triv = load_experiment_result(
            str(triv_files[subject]), pred_key=pred_key
        )
        y_true_triv, y_pred_triv = r_triv.get_all_trials()
        trial_ids_triv = r_triv.get_trial_indices()

        validate_trial_alignment(
            trial_ids_main, trial_ids_triv,
            name_a="Video model", name_b=label,
        )

        if not np.allclose(y_true, y_true_triv, equal_nan=True):
            raise ValueError(
                f"S{subject:02d} / {label}: y_true differs from Video model."
            )

        trivial_preds[label] = y_pred_triv

    results[subject] = {
        "y_true":             y_true,
        "trial_ids":          trial_ids_main,
        "y_true_shuf_permuted": y_true_shuf_permuted,
        "pred": {
            "Video model":                y_pred_main,
            "Shuffled model":             y_pred_shuf,
            "Training-set mean waveform": trivial_preds["Training-set mean waveform"],
            "Training-mean constant":     trivial_preds["Training-mean constant"],
            "Zero-output baseline":       trivial_preds["Zero-output baseline"],
        },
    }

print("All conditions loaded and aligned.")


## 5. Shuffle audit (internal QC)
Verifies the shuffled pipeline really broke video–EEG trial correspondence.

In [ ]:
shuffle_audit_rows = []
for subject in SUBJECTS:
    y_orig = results[subject]["y_true"]
    y_permuted = results[subject]["y_true_shuf_permuted"]

    n_total = y_orig.size
    n_diff  = int(np.sum(~np.isclose(y_orig, y_permuted, atol=1e-12,
                                    equal_nan=True)))

    shuffle_audit_rows.append({
        "Subject":            subject,
        "N samples":          n_total,
        "N differing":        n_diff,
        "Fraction differing": n_diff / n_total,
        "Mean |y_orig − y_permuted|": float(np.mean(np.abs(y_orig - y_permuted))),
    })

df_shuffle_audit = pd.DataFrame(shuffle_audit_rows)
display(df_shuffle_audit.round(4))

frac_min = df_shuffle_audit["Fraction differing"].min()
assert frac_min > 0.99, (
    f"Shuffling did not substantially modify the EEG targets "
    f"(min fraction differing = {frac_min:.4f})."
)

## 6. Trial-level statistical summaries (all 6 conditions)

In [ ]:
subject_summaries = {}

for subject in SUBJECTS:
    y_orig   = results[subject]["y_true"]
    y_permuted = results[subject]["y_true_shuf_permuted"]
    preds    = results[subject]["pred"]

    subject_summaries[subject] = {}
    for cond in CONDITIONS:
        pred_key, target_key = CONDITION_SPEC[cond]
        y_target = y_orig if target_key == "original" else y_permuted
        subject_summaries[subject][cond] = summarize_trials(
            y_target, preds[pred_key], seed=SEED, n_boot=N_BOOT
        )

print(f"Trial-level summaries computed for {len(CONDITIONS)} conditions "
      f"× {len(SUBJECTS)} subjects.")


## 7. Subject-level performance table (internal; export commented)


In [ ]:
rows = []
for subject in SUBJECTS:
    for condition in CONDITIONS:
        s       = subject_summaries[subject][condition]
        r_ci    = s["r_pooled_ci_boot"]
        rmse_ci = s["rmse_trial_ci_boot"]

        rows.append({
            "Subject":             subject,
            "Condition":           condition,
            "N trials":            s["trials"],
            "Fisher-z pooled r":   s["r_pooled"],
            "r CI low (trial)":    r_ci[0],
            "r CI high (trial)":   r_ci[1],
            "Trial-level RMSE":    s["rmse_mean"],
            "RMSE CI low (trial)": rmse_ci[0],
            "RMSE CI high (trial)": rmse_ci[1],
        })

df_subject = pd.DataFrame(rows)
display(df_subject.round(4))

# ## 8. Within-subject paired comparisons

Video model vs each of the 5 null conditions, aggregated over the
same 40 held-out trial positions per participant. For the shuffled-
pairing model under permuted-target scoring, the EEG reference array
differs from the video-model condition; the contrast is reported
separately and is not interpreted as a like-for-like comparison.

In [ ]:
subject_tests = {}
for subject in SUBJECTS:
    subject_tests[subject] = {}
    for condition in CONDITIONS[1:]:
        subject_tests[subject][condition] = compare_methods_trial_level(
            subject_summaries[subject]["Video model"],
            subject_summaries[subject][condition],
            method_names=("Video model", condition),
            seed=SEED,
            n_perm=N_PERM,
            n_boot=N_BOOT,
        )

print("Within-subject paired trial-level comparisons completed.")

## 9. Within-subject paired effects table (internal; export commented)


In [ ]:
def _ci_bounds(ci):
    if isinstance(ci, dict):
        return ci["low"], ci["high"]
    return ci[0], ci[1]

within_rows = []
for subject in SUBJECTS:
    for condition in CONDITIONS[1:]:
        test = subject_tests[subject][condition]
        r_s  = test["r"]["summary"]
        e_s  = test["rmse"]["summary"]

        r_ci_low,  r_ci_high  = _ci_bounds(r_s["ci_boot"])
        e_ci_low,  e_ci_high  = _ci_bounds(e_s["ci_boot"])

        within_rows.append({
            "Subject":    subject,
            "Comparison": f"Video model − {condition}",
            "Delta_z_r":            r_s["mean"],
            "Delta_z_r_CI_low":     r_ci_low,
            "Delta_z_r_CI_high":    r_ci_high,
            "Delta_z_r_p":          r_s["tests"]["sign_perm_p"],
            "Delta_r":              test["r"].get("delta_mean_r", np.nan),
            "Delta_RMSE":           e_s["mean"],
            "Delta_RMSE_CI_low":    e_ci_low,
            "Delta_RMSE_CI_high":   e_ci_high,
            "Delta_RMSE_p":         e_s["tests"]["sign_perm_p"],
        })

df_within_subject_effects = pd.DataFrame(within_rows)
display(df_within_subject_effects.round(4))


## 10. Population-level aggregation helper (identical to Section 4.4)

In [ ]:
def _aggregate_population(values, scale="linear",
                          n_boot=N_BOOT, ci_level=CI_LEVEL, seed=SEED):
    v = np.asarray(values, dtype=float)
    v = v[np.isfinite(v)]
    n = v.size
    if n == 0:
        return dict(n=0, mean=np.nan, ci_lo=np.nan, ci_hi=np.nan)

    v_t    = fisher_z(v) if scale == "fisher" else v
    mean_t = float(np.mean(v_t))

    if n > 1:
        boot = stats.bootstrap(
            (v_t,), np.mean,
            confidence_level=ci_level,
            n_resamples=n_boot,
            method="percentile",
            random_state=seed,
        )
        ci_lo_t = float(boot.confidence_interval.low)
        ci_hi_t = float(boot.confidence_interval.high)
    else:
        ci_lo_t = ci_hi_t = mean_t

    if scale == "fisher":
        mean_disp  = float(inv_fisher_z(mean_t))
        ci_lo_disp = float(inv_fisher_z(ci_lo_t))
        ci_hi_disp = float(inv_fisher_z(ci_hi_t))
    else:
        mean_disp, ci_lo_disp, ci_hi_disp = mean_t, ci_lo_t, ci_hi_t

    return dict(n=n, mean=mean_disp, ci_lo=ci_lo_disp, ci_hi=ci_hi_disp)


def _effect_stats(values, scale, n_perm=N_PERM, seed=SEED):
    """Paired-effect statistics across 18 participants."""
    v = np.asarray(values, dtype=float)
    v = v[np.isfinite(v)]

    if v.size == 0:
        return {
            "N": 0, "mean": np.nan, "CI_low": np.nan, "CI_high": np.nan,
            "signflip_p": np.nan, "wilcoxon_p": np.nan, "Cohen_dz": np.nan,
        }

    agg = _aggregate_population(v, scale=scale)

    # ---- Transform to the inferential scale ----
    # For condition-level correlations, aggregation and inference are
    # performed in Fisher-z space (see Validation Methods). For paired
    # contrasts (Δz), values are already on the Fisher-z scale, so
    # scale="linear" leaves them unchanged.
    v_test = fisher_z(v) if scale == "fisher" else v

    p_perm = float(sign_flip_pvalue(v_test, n_perm=n_perm, seed=seed))
    try:
        p_wil = float(stats.wilcoxon(
            v_test, alternative="two-sided", zero_method="wilcox"
        ).pvalue)
    except ValueError:
        p_wil = np.nan

    sd = float(np.std(v_test, ddof=1))
    dz = float(np.mean(v_test) / sd) if sd > 0 else np.nan

    return {
        "N":           agg["n"],
        "mean":        agg["mean"],
        "CI_low":      agg["ci_lo"],
        "CI_high":     agg["ci_hi"],
        "signflip_p":  p_perm,
        "wilcoxon_p":  p_wil,
        "Cohen_dz":    dz,
    }


def _one_sample_vs_ref(values, ref, n_perm=N_PERM, seed=SEED):
    """One-sample sign-flip test against a fixed reference."""
    v = np.asarray(values, dtype=float)
    v = v[np.isfinite(v)]
    if v.size == 0:
        return dict(N=0, mean=np.nan, CI_low=np.nan, CI_high=np.nan,
                    ref=ref, signflip_p=np.nan,
                    wilcoxon_p=np.nan, Cohen_dz=np.nan)

    agg = _aggregate_population(v, scale="linear")
    d = v - ref
    mean_d = float(np.mean(d))
    sd_d   = float(np.std(d, ddof=1)) if v.size > 1 else np.nan
    dz     = mean_d / sd_d if (np.isfinite(sd_d) and sd_d > 0) else np.nan

    p_perm = float(sign_flip_pvalue(d, n_perm=n_perm, seed=seed))
    try:
        p_wil = float(stats.wilcoxon(
            d, alternative="two-sided", zero_method="wilcox"
        ).pvalue)
    except ValueError:
        p_wil = np.nan

    return dict(
        N=agg["n"], mean=agg["mean"],
        CI_low=agg["ci_lo"], CI_high=agg["ci_hi"],
        ref=float(ref),
        signflip_p=p_perm, wilcoxon_p=p_wil,
        Cohen_dz=float(dz) if np.isfinite(dz) else np.nan,
    )


## 11. Table 1 — Population-level condition summary
For each condition: r (Fisher-z), RMSE, 95% CI, p, d_z.

One-sample tests: r vs 0; RMSE vs 1.0.

In [ ]:
condition_rows = []

for condition in CONDITIONS:

    # ---- r ----
    r_values = np.array(
        [subject_summaries[s][condition]["r_pooled"] for s in SUBJECTS],
        dtype=float,
    )
    r_valid = r_values[np.isfinite(r_values)]

    if r_valid.size > 0:
        r_pop = _aggregate_population(r_valid, scale="fisher")
        r_eff = _effect_stats(r_valid, scale="fisher")
    else:
        r_pop = dict(n=0, mean=np.nan, ci_lo=np.nan, ci_hi=np.nan)
        r_eff = dict(N=0, signflip_p=np.nan,
                     wilcoxon_p=np.nan, Cohen_dz=np.nan)

    # ---- RMSE ----
    rmse_values = np.array(
        [subject_summaries[s][condition]["rmse_mean"] for s in SUBJECTS],
        dtype=float,
    )
    rmse_valid = rmse_values[np.isfinite(rmse_values)]
    rmse_eff   = _one_sample_vs_ref(rmse_valid, ref=RMSE_REF)

    condition_rows.append({
        "Condition":              condition,
        "N (r)":                  r_pop["n"],
        "r (Fisher-z)":           r_pop["mean"],
        "r CI low":               r_pop["ci_lo"],
        "r CI high":              r_pop["ci_hi"],
        "r signflip p":           r_eff["signflip_p"],
        "r wilcoxon p":           r_eff["wilcoxon_p"],
        "r Cohen dz":             r_eff["Cohen_dz"],
        "N (RMSE)":               rmse_eff["N"],
        "RMSE":                   rmse_eff["mean"],
        "RMSE CI low":            rmse_eff["CI_low"],
        "RMSE CI high":           rmse_eff["CI_high"],
        "RMSE ref":               rmse_eff["ref"],
        "RMSE signflip p":        rmse_eff["signflip_p"],
        "RMSE wilcoxon p":        rmse_eff["wilcoxon_p"],
        "RMSE Cohen dz":          rmse_eff["Cohen_dz"],
    })

df_pop_conditions = pd.DataFrame(condition_rows)

# Holm within r family (only defined-r rows)
mask_r = df_pop_conditions["N (r)"].gt(0)
p_r    = df_pop_conditions.loc[mask_r, "r signflip p"].to_numpy(float)
adj_r  = np.full(len(p_r), np.nan)
if np.isfinite(p_r).any():
    adj_r = multipletests(p_r, method="holm")[1]
df_pop_conditions["r Holm p"] = np.nan
df_pop_conditions.loc[mask_r, "r Holm p"] = adj_r

# Holm within RMSE family
p_e   = df_pop_conditions["RMSE signflip p"].to_numpy(float)
adj_e = np.full(len(p_e), np.nan)
if np.isfinite(p_e).any():
    adj_e = multipletests(p_e, method="holm")[1]
df_pop_conditions["RMSE Holm p"] = adj_e

display(df_pop_conditions.round(4))


## 12. Table 2 — Paired contrasts (Video model − each null)

In [ ]:
effect_rows = []

for condition in CONDITIONS[1:]:

    # ---- Δz(r) ----
    r_effects = np.array(
        [subject_tests[s][condition]["r"]["summary"]["mean"] for s in SUBJECTS],
        dtype=float,
    )
    effect_rows.append({
        "Comparison": f"Video model − {condition}",
        "Metric":     "Fisher-z correlation",
        **_effect_stats(r_effects, scale="linear"),
    })

    # ---- ΔRMSE ----
    rmse_effects = np.array(
        [subject_tests[s][condition]["rmse"]["summary"]["mean"] for s in SUBJECTS],
        dtype=float,
    )
    effect_rows.append({
        "Comparison": f"Video model − {condition}",
        "Metric":     "RMSE",
        **_effect_stats(rmse_effects, scale="linear"),
    })

df_pop_effects = pd.DataFrame(effect_rows)

# Holm within each metric family
df_pop_effects["Holm-adjusted p"] = np.nan
for metric in ("Fisher-z correlation", "RMSE"):
    mask = df_pop_effects["Metric"].eq(metric)
    p    = df_pop_effects.loc[mask, "signflip_p"].to_numpy(dtype=float)
    valid  = np.isfinite(p)
    adj    = np.full(len(p), np.nan)
    if valid.any():
        adj[valid] = multipletests(p[valid], method="holm")[1]
    df_pop_effects.loc[mask, "Holm-adjusted p"] = adj

display(df_pop_effects.round(4))

## 13. df_group — DataFrame for plot_null_models_summary
Column names must match those expected by the plotting function.

In [ ]:
group_rows = []
for _, row in df_pop_conditions.iterrows():
    group_rows.append({
        "Condition":                row["Condition"],
        "N subjects (r)":           row["N (r)"],
        "Participant-aggregated r": row["r (Fisher-z)"],
        "r CI Low":                 row["r CI low"],
        "r CI High":                row["r CI high"],
        "N subjects (RMSE)":        row["N (RMSE)"],
        "Mean participant-level RMSE":  row["RMSE"],
        "RMSE CI Low":              row["RMSE CI low"],
        "RMSE CI High":             row["RMSE CI high"],
    })

df_group = pd.DataFrame(group_rows)
display(df_group.round(4))


## 14. Main null-model figure

In [ ]:
fig, axes = plot_null_models_summary(
    df_subject,
    df_group=df_group,
    save_fig=True,
    file_name=str(FIG_DIR / "exp3_null_models.pdf"),
)
plt.show()
plt.close(fig)

## 15. Export tables
Population tables are exported. Subject-level tables are commented.

In [ ]:
# --- Population tables (Table 1 and Table 2) ---
df_pop_conditions.to_csv(
    TABLE_DIR / "exp3_population_conditions.csv", index=False
)
df_pop_effects.to_csv(
    TABLE_DIR / "exp3_population_effects.csv", index=False
)
df_group.to_csv(
    TABLE_DIR / "exp3_population_summary.csv", index=False
)

# --- Per-subject tables (commented out) ---
# df_subject.to_csv(
#     TABLE_DIR / "exp3_subject_level.csv", index=False
# )
# df_within_subject_effects.to_csv(
#     TABLE_DIR / "exp3_within_subject_effects.csv", index=False
# )
# df_shuffle_audit.to_csv(
#     TABLE_DIR / "exp3_shuffle_audit.csv", index=False
# )

print("Population tables exported to:", TABLE_DIR)